In [1]:
import os

In [2]:
%pwd


'e:\\Udemy\\datascienceproject\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\Udemy\\datascienceproject'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path


In [6]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

e:\Udemy\datascienceproject\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
from pathlib import Path

CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")
SCHEMA_FILE_PATH = Path("schema.yaml")


In [ ]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)
    
        create_directories([Path(self.config.artifacts_root)])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config


In [24]:
## component-Data Ingestion
from venv import logger
import urllib.request as request
from flask import request
import zipfile
import shutil



class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.warning(f"File already exists of path: {self.config.local_data_file}")
    
    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        if not os.path.exists(unzip_path):
            os.makedirs(unzip_path, exist_ok=True)
            shutil.unpack_archive(
                filename=self.config.local_data_file,
                extract_dir=unzip_path
            )
            logger.info(f"File extracted at location: {unzip_path}")
        else:
            logger.warning(f"File already exists of path: {unzip_path}")



In [25]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    print(data_ingestion_config)

    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    logger.exception(e)

[2026-06-05 15:08:42,126: INFO: common]: Successfully read YAML file: config\config.yaml
[2026-06-05 15:08:42,130: INFO: common]: Successfully read YAML file: params.yaml
[2026-06-05 15:08:42,138: INFO: common]: Successfully read YAML file: schema.yaml
[2026-06-05 15:08:42,143: INFO: common]: Directory created successfully: artifacts
[2026-06-05 15:08:42,145: ERROR: 3073152026]: isinstance() arg 2 must be a type, a tuple of types, or a union
Traceback (most recent call last):
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_28692\3073152026.py", line 2, in <module>
    config = ConfigurationManager()
  File "C:\Users\LENOVO\AppData\Local\Temp\ipykernel_28692\2214064951.py", line 10, in __init__
    create_directories([self.config.artifacts_root])
  File "e:\Udemy\datascienceproject\venv\lib\site-packages\ensure\main.py", line 873, in __call__
    if not isinstance(return_val, self.return_templ):
TypeError: isinstance() arg 2 must be a type, a tuple of types, or a union
